# 作业一
参考sql agent，实现一下基于 chinook.db 数据集进行问答agent（nl2sql），需要能回答如下提问：
- 提问1: 数据库中总共有多少张表；
- 提问2: 员工表中有多少条记录
- 提问3: 在数据库中所有客户个数和员工个数分别是多少

# 数据库解析

In [2]:
import sqlite3

conn = sqlite3.connect("chinook.db")

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
tables

[('albums',),
 ('sqlite_sequence',),
 ('artists',),
 ('customers',),
 ('employees',),
 ('genres',),
 ('invoices',),
 ('invoice_items',),
 ('media_types',),
 ('playlists',),
 ('playlist_track',),
 ('tracks',),
 ('sqlite_stat1',)]

In [3]:
from typing import Union
import traceback
from sqlalchemy import create_engine, inspect, func, select, Table, MetaData
import pandas as pd


class MyDB:
    def __init__(self, db_url: str) -> None:

        if 'sqlite' in db_url:
            self.db_type = 'sqlite'
        elif 'mysql' in db_url:
            self.db_type = 'mysql'

        self.engine = create_engine(db_url, echo=False)
        self.conn = self.engine.connect()
        self.db_url = db_url

        self.inspector = inspect(self.engine)  # 创建一个检查器对象(inspector-检查督察)，用于获取数据库的元数据
        self.table_names = self.inspector.get_table_names()

        self._table_fields = {}
        self.foreign_keys = []
        self._table_sample = {}

        for table_name in self.table_names:
            print('Table Name:', table_name)
            self._table_fields[table_name] = {}

            self.foreign_keys += [
                {
                    'constrained_table': table_name,
                    'constrained_columns': x['constrained_columns'],
                    'referred_table': x['referred_table'],
                    'referred_columns': x['referred_columns']
                } for x in self.inspector.get_foreign_keys(table_name)
            ]

            table_instance = Table(table_name, MetaData(), autoload_with=self.engine)
            table_columns = self.inspector.get_columns(table_name)
            self._table_fields[table_name] = {x['name']: x for x in table_columns}

            for column_meta in table_columns:

                column_instance = getattr(table_instance.columns, column_meta['name'])

                query = select(func.count(func.distinct(column_instance)))
                distinct_count = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['distinct_count'] = distinct_count

                field_type = self._table_fields[table_name][column_meta['name']]['type']
                field_type = str(field_type)
                if 'text' in field_type.lower() or 'char' in field_type.lower():
                    query = (select(column_instance, func.count().label('count'))
                             .group_by(column_instance)
                             .order_by(func.count().desc())
                             .limit(1)
                             )

                    top1_value = self.conn.execute(query).fetchone()[0]
                    self._table_fields[table_name][column_meta['name']]['mode'] = top1_value

                query = select(func.count()).filter(column_instance == None)
                nan_count = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['nan_count'] = nan_count

                query = select(func.max(column_instance))
                max_value = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['max'] = max_value

                query = select(func.min(column_instance))
                min_value = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['min'] = min_value

                query = select(column_instance).limit(10)
                random_value = self.conn.execute(query).all()
                random_value = [x[0] for x in random_value]
                random_value = [str(x) for x in random_value if x is not None]
                random_value = list(set(random_value))
                self._table_fields[table_name][column_meta['name']]['random_value'] = random_value[:3]

            query = select(table_instance)
            self._table_sample[table_name] = pd.DataFrame([self.conn.execute(query).fetchone()])
            self._table_sample[table_name].columns = [x['name'] for x in table_columns]

    def get_table_fields(self, table_name: str) -> pd.DataFrame:
        return pd.DataFrame.from_dict(self._table_fields[table_name]).T

    def get_data_relations(self) -> pd.DataFrame:
        return pd.DataFrame(self.foreign_keys)

    def get_table_sample(self, table_name: str) -> pd.DataFrame:
        return self._table_sample[table_name]

    def check_sql(self, sql) -> Union[bool, str]:
        try:
            self.engine.execute(sql)
            return True, 'ok'
        except:
            err_msg = traceback.format_exc()
            return False, err_msg

    def execute_sql(self, sql) -> bool:
        result = self.engine.execute(sql)
        return list(result)



In [4]:
parser = MyDB('sqlite:///./chinook.db')

Table Name: albums
Table Name: artists
Table Name: customers
Table Name: employees
Table Name: genres
Table Name: invoice_items
Table Name: invoices
Table Name: media_types
Table Name: playlist_track
Table Name: playlists
Table Name: tracks


In [5]:
parser.get_data_relations()

,constrained_table,constrained_columns,referred_table,referred_columns
0,albums,[ArtistId],artists,[ArtistId]
1,customers,[SupportRepId],employees,[EmployeeId]
2,employees,[ReportsTo],employees,[EmployeeId]
3,invoice_items,[TrackId],tracks,[TrackId]
4,invoice_items,[InvoiceId],invoices,[InvoiceId]
5,invoices,[CustomerId],customers,[CustomerId]
6,playlist_track,[TrackId],tracks,[TrackId]
7,playlist_track,[PlaylistId],playlists,[PlaylistId]
8,tracks,[MediaTypeId],media_types,[MediaTypeId]
9,tracks,[GenreId],genres,[GenreId]


In [6]:
parser.get_table_sample('albums')

,AlbumId,Title,ArtistId
0,1,For Those About To Rock We Salute You,1


In [7]:
parser.get_table_fields('genres')

,name,type,nullable,default,primary_key,distinct_count,nan_count,max,min,random_value,mode
GenreId,GenreId,INTEGER,False,None,1,25,0,25,1,"[7, 4, 1]",NaN
Name,Name,NVARCHAR(120),True,None,0,25,0,World,Alternative,"[Latin, Soundtrack, Blues]",World


# jwt Token生成

In [8]:
import time
import jwt

def generate_token(apikey:str,exp_seconds:int):
    try:
        id,secret = apikey.split('.')
        payload = {
            'api_key':id,
            'exp':int(round(time.time() * 1000)) + exp_seconds * 1000,
            'timestamp':int(round(time.time() * 1000))
        }
        return jwt.encode(payload, secret, algorithm='HS256',headers={'alg': 'HS256','sign_type':'SIGN'})
    except ValueError:
        return apikey

# 大模型调用方法

In [9]:
import os
import requests
from langchain_openai import ChatOpenAI

def ask_qwen(question,apikey,nretry=5):
    if nretry == 0:
        return None
    
    token = generate_token(apikey,exp_seconds=3600)
    url = "https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions"
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {token}'
    }
    data = {
        'model':'qwen3-max',
        'messages':[
            {
                'role':'user',
                'content':question
            }
        ],
        'temperature':0.5
    }

    try:
        response = requests.post(url, headers=headers, json=data,timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f'请求报错，正在重试...剩余重试次数{nretry-1}，错误信息：{e}')
        return ask_qwen(question, apikey, nretry - 1)

def ask_qwen_langchain(question,apikey):
    token = generate_token(apikey,exp_seconds=3600)
    
    # 设置环境变量供 LangChain 使用
    os.environ["OPENAI_API_KEY"] = token
    os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"

    llm = ChatOpenAI(
        model = "qwen3-max",
        base_url=os.environ["OPENAI_BASE_URL"],
        api_key=token,
        streaming=True
    )

    print("📝 [LangChain] 流式回复: ", end="", flush=True)
    for chunk in llm.stream(question):
        print(chunk.content,end='',flush=True)
    print()

E:\MiniConda\Installation\envs\py312_1\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [10]:
# 针对整个数据库架构的提示词
global_prompt = '''你是一个专业的数据库SQL工程师。以下是我们数据库的整体结构和字段信息：

数据库中的表及其外键关联关系：
{relations}

各表的Schema信息字典（包含字段名，类型，主要用于参考）：
{all_schemas}

请你根据上面的信息和我的提问，写出对应的SQLite SQL查询语句。
注意：
1. 仅输出SQL语句，千万不要包含markdown包裹符号（例如不要输出 ```sql ... ```），也不要有任何额外的解释，直接返回纯SQL代码即可！
2. 即使问题中询问总共有多少表之类的信息，也必须写出动态去数据库或其系统表(如 sqlite_master)中查询的SQL语句(例如 SELECT count(name) FROM sqlite_master WHERE type='table')，千万不能直接输出硬编码的数字答案。

我的提问是：{question}
'''

answer_rewrite_prompt = '''你是一个专业的数据库专家，将下面的问题回答组织为自然语言。：

原始问题：{question}

执行SQL：{sql}

原始结果：{answer}
'''


In [11]:
import re

def nl2sql_and_execute(question, apikey, parser):
    # 1. 组装数据库所有表结构和关系信息
    relations = parser.get_data_relations().to_markdown()
    
    schemas = []
    for table_name in parser.table_names:
        schema_md = f"---表 {table_name} Schema---\n"
        # 获取表结构
        schema_df = parser.get_table_fields(table_name)
        schema_md += schema_df[['type', 'distinct_count']].to_markdown()
        schemas.append(schema_md)
        
    all_schemas = "\n\n".join(schemas)

    # print("数据库表关系:\n", relations)
    # print("\n数据库表结构和字段信息:\n", all_schemas)
    
    # 2. 格式化提示词
    current_prompt = global_prompt.format(
        relations=relations,
        all_schemas=all_schemas,
        question=question
    )
    
    print(f"----- 问题：{question} -----")
    
    # 3. 调用大模型生成SQL
    # 我们使用前面定义好的 ask_qwen 函数
    res = ask_qwen(current_prompt, apikey)
    
    # 解析模型返回的答案
    sql = res['choices'][0]['message']['content'].strip()
    
    # 清理可能被包裹的Markdown代码块
    if sql.startswith('```sql'):
        sql = sql[6:]
    if sql.startswith('```'):
        sql = sql[3:]
    if sql.endswith('```'):
        sql = sql[:-3]
    sql = sql.strip()
    
    print(f"生成SQL: {sql}")
    
    # 4. 在数据库中执行SQL
    try:
        from sqlalchemy import text
        # 虽然parser里面有execute_sql, 但可能存在拼写错误, 因此这里直接使用sqlalchemy text包执行更为稳妥
        with parser.engine.connect() as conn:
            result = conn.execute(text(sql)).fetchall()
            print(f"执行结果: {result}\n")

        # 解决报错：不能用同名的变量名赋值并格式化覆盖外部全局变量，改名即可
        formatted_answer_prompt = answer_rewrite_prompt.format(
            question=question,
            sql=sql,
            answer=result
        )
        
        # 5. 将SQL执行结果组织为自然语言
        res_nl = ask_qwen(formatted_answer_prompt, apikey)
        answer = res_nl['choices'][0]['message']['content'].strip()
        print(f"自然语言答案: {answer}\n")
        
    except Exception as e:
        print(f"执行SQL报错: {e}\n")

In [12]:
if __name__ == "__main__":
    ALIYUN_API_KEY = "sk-e0e141c7108a4cabaf4cced1f749ae89"
    
    # 实例化我们的数据库解析器
    parser = MyDB('sqlite:///./chinook.db')
    
    # 待解答的问题列表
    questions = [
        "数据库中总共有多少张表；",
        "员工表中有多少条记录",
        "在数据库中所有客户个数和员工个数分别是多少"
    ]
    
    # 遍历每个问题并执行NL2SQL流程
    for q in questions:
        nl2sql_and_execute(q, ALIYUN_API_KEY, parser)

Table Name: albums
Table Name: artists
Table Name: customers
Table Name: employees
Table Name: genres
Table Name: invoice_items
Table Name: invoices
Table Name: media_types
Table Name: playlist_track
Table Name: playlists
Table Name: tracks
----- 问题：数据库中总共有多少张表； -----
生成SQL: SELECT count(name) FROM sqlite_master WHERE type='table';
执行结果: [(13,)]

自然语言答案: 数据库中共有13张表。

----- 问题：员工表中有多少条记录 -----
生成SQL: SELECT COUNT(*) FROM employees;
执行结果: [(8,)]

自然语言答案: 员工表中共有8条记录。

----- 问题：在数据库中所有客户个数和员工个数分别是多少 -----
生成SQL: SELECT 
  (SELECT COUNT(*) FROM customers) AS customer_count,
  (SELECT COUNT(*) FROM employees) AS employee_count;
执行结果: [(59, 8)]

自然语言答案: 数据库中共有 59 位客户和 8 名员工。



# 作业二
阅读 06-stock-bi-agent 代码，回答如下问题：
- 什么是前后端分离？
- 历史对话如何存储，以及如何将历史对话作为大模型的下一次输入；

什么是前后端分离？

答：前后端分离是一种软件架构模式，将用户界面(前端)和业务逻辑/数据处理(后端)拆分为两个独立的应用程序，通过HTTP API进行通信
1. 前端：用户输入"分析苹果公司股票"
         ↓
2. 前端：st.chat_input() 捕获输入
         ↓
3. 前端：requests.post("http://127.0.0.1:8000/v1/chat/",
                      json={"content": "分析苹果公司股票",
                            "session_id": "abc123",
                            "tools": ["get_stock_info"]})
         ↓
4. 后端：FastAPI接收请求 → routers/chat.py
         ↓
5. 后端：调用 services/chat.py 处理业务逻辑
         ↓
6. 后端：调用 OpenAI Agent → 大模型生成回答
         ↓
7. 后端：通过SSE流式返回回答片段
         ↓
8. 前端：async for data in response:
             placeholder.markdown(data)  # 实时显示


历史对话如何存储，以及如何将历史对话作为大模型的下一次输入

答：使用业务数据库 "server.db" 存储会话管理和用户查询，使用 Agent状态数据库 "conversations.db" 用于OpenAI Agent SDK自动管理对话上下文

第1轮对话：
────────────────────────────────────────
用户："分析苹果公司股票"

1. 存储到sever.db：
   - ChatMessageTable插入: role="user", content="分析苹果公司股票"

2. 调用大模型：
   session = AdvancedSQLiteSession(session_id="abc123")

   Runner.run_streamed(
       agent,
       input="分析苹果公司股票",  # 当前输入
       session=session  # 此时session为空（首次对话）
   )

3. 大模型收到：
   [
     {"role": "system", "content": "你是股票分析助手..."},
     {"role": "user", "content": "分析苹果公司股票"}
   ]

4. 大模型返回："苹果公司当前股价..."

5. 存储到sever.db：
   - ChatMessageTable插入: role="assistant", content="苹果公司当前股价..."

6. AdvancedSQLiteSession自动记录：
   - conversations.db中保存了完整的对话历史


第2轮对话：
────────────────────────────────────────
用户："它的PE是多少？"

1. 存储到sever.db：
   - ChatMessageTable插入: role="user", content="它的PE是多少？"

2. 调用大模型：
   session = AdvancedSQLiteSession(session_id="abc123")

   Runner.run_streamed(
       agent,
       input="它的PE是多少？",  # 当前输入
       session=session  # 此时session包含第1轮历史！
   )

3. 大模型收到（自动拼接历史）：
   [
     {"role": "system", "content": "你是股票分析助手..."},
     {"role": "user", "content": "分析苹果公司股票"},
     {"role": "assistant", "content": "苹果公司当前股价..."},
     {"role": "user", "content": "它的PE是多少？"}  # 当前输入
   ]

4. 大模型理解上下文：
   - 知道"它"指的是苹果公司
   - 返回："苹果当前PE为28.5..."

5. 存储到sever.db：
   - ChatMessageTable插入: role="assistant", content="苹果当前PE为28.5..."
